# In-class Session: Generate and Validate New Molecules and Crystals

AI for Materials Science — Hands-on session 8

A generative model generates new candidates, but a candidate is only useful after it has been validated. Today you do
both, twice:

1. **Molecules:** train a graph variational autoencoder (VAE) on QM9 molecules, sample new molecules from it, and
   measure how many are valid, unique and new.
2. **Crystals:** use **MatterGen**, a pretrained diffusion model, to generate Li–P–S crystals for sulfide solid
   electrolytes, and validate them with four criteria, one of which is a match with a known Materials Project phase.

**Before you run any cell, switch to a T4 GPU:** *Runtime → Change runtime type → T4 GPU*. Do it first, because
changing the runtime type starts a new session and loses everything installed or run so far. The GPU is only for
MatterGen; the molecule model trains on the CPU either way. If Colab gives you no GPU, continue on a CPU runtime.
Without a GPU, or if your MatterGen run fails, you use structures that the TA generated with the same model,
conditions and seed, so you can still finish every section.

## Today's plan

- **0. Setup:** install the libraries, download the data and, on a GPU runtime, start installing MatterGen.
- **A. Molecule data:** QM9 molecules as adjacency tensors and feature matrices.
- **B. Graph VAE:** build the model in PyTorch and train it on a CPU.
- **C. Generate and validate molecules:** validity, uniqueness, novelty, and Practice 1.
- **D. MatterGen:** how a diffusion model generates a crystal, conditions, guidance, your own run on a GPU, and the
  TA's structures of a baseline and a conditioned run.
- **E. Validate crystals:** four validation criteria, a comparison with the baseline, and Practice 2.
- **F. Assignment:** does the condition change what MatterGen makes? Two files to submit.

## 0. Setup

### 0-1. Install the libraries

Run this cell first. It installs RDKit (molecules), pymatgen (crystal structures) and ASE (structure plots). Colab already includes PyTorch, NumPy, pandas, scikit-learn and Matplotlib.

In [ ]:
!pip install -q rdkit==2026.9.1 pymatgen==2026.10.24 ase==3.29.0

### 0-2. Download the course material

This cell copies the course repository into a folder called `MS49900-AI4M`. The notebook reads these files from its `Data/` folder:

- `qm9_subset_20k.csv`: 20,000 QM9 molecules with their HOMO–LUMO gap (sections A–C).
- `mp_LiPS_references.json.gz`: known Li–P–S(–Cl, –Br) crystals from the Materials Project (section E).
- `mattergen_pregenerated/`: the structures the TA generated with MatterGen (sections D–F).
- `mattergen_lock.txt`: the exact MatterGen package versions, used by 0-5 on a GPU runtime.

If git says the destination already exists, it leaves that folder unchanged and does not update the files. If a later cell reports a missing file, check the contents of `MS49900-AI4M/Data/`.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

The next cell imports the tools used throughout the notebook.

In [ ]:
import gzip, hashlib, json, math, os, subprocess, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.container import BarContainer
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from rdkit import Chem, RDLogger
from rdkit.Chem import BondType, Descriptors, Draw, QED
from pymatgen.core import Composition, Structure
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize.plot import plot_atoms
RDLogger.DisableLog("rdApp.*")

### 0-4. Set the file paths

`DATA_DIR` points to the downloaded data, `PREGEN` to the TA's MatterGen structures, and `OUTPUT` to the folder where the notebook saves your submission files. It should print `Data folder exists: True`; if not, check the output of 0-2.

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
PREGEN = DATA_DIR / "mattergen_pregenerated"
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
torch.manual_seed(0)
print("Data folder exists:", DATA_DIR.exists())

### 0-5. On a GPU runtime: start installing MatterGen in the background

On a GPU runtime this cell installs MatterGen in the background (2–5 minutes), with the exact package versions in
`mattergen_lock.txt`. MatterGen gets its own Python 3.10 environment (`/content/mg_env`), as in its installation
instructions, because it needs older versions of PyTorch (2.2) and NumPy (below 2.0) than the rest of this notebook
uses. The separate environment keeps both sets of versions working; D-2 runs MatterGen from it as a separate
program. Look for `GPU found. MatterGen installation: running`, then go on with the molecules; D-2 checks whether the
installation has finished. Running the cell again only reports the status;
it never starts a second installation. Without a GPU it only prints a message: you use the TA's structures, made with
the same model, conditions and seed.

In [ ]:
MODEL, SEED, BATCH_SIZE = "chemical_system_energy_above_hull", 42, 8
MATTERGEN_COMMIT = "92423660a8bd70e83679086e88f88596d484dc16"
MG = "/content/mg_env/bin"
USE_GPU = torch.cuda.is_available()
JOBS = globals().get("JOBS", {})                 ## background jobs survive a rerun of this cell

def job_state(name):
    job = JOBS.get(name)
    if job is None:
        return "not started"
    rc = job["proc"].poll()
    return "running" if rc is None else ("done" if rc == 0 else f"failed (exit code {rc}, see {job['log']})")

def start_job(name, cmd, log, **popen_args):
    """Start a background job once: if it is running or has finished, return it instead of starting another."""
    if job_state(name) in ("running", "done"):
        return JOBS[name]
    proc = subprocess.Popen(cmd, stdout=open(log, "a"), stderr=subprocess.STDOUT, **popen_args)  ## keeps failed logs
    JOBS[name] = {"proc": proc, "started": time.time(), "log": log}
    return JOBS[name]

if USE_GPU:
    pip_args = "--extra-index-url https://download.pytorch.org/whl/cu118 --find-links https://data.pyg.org/whl/torch-2.2.0+cu118.html --index-strategy unsafe-best-match"
    py = "--python /content/mg_env/bin/python"
    lock = DATA_DIR / "mattergen_lock.txt"
    steps = ["pip -q install uv", "rm -rf /content/mg_env /content/mattergen",
             "GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/microsoft/mattergen /content/mattergen",
             f"git -C /content/mattergen checkout -q {MATTERGEN_COMMIT}", "uv venv /content/mg_env --python 3.10 -q"]
    if lock.exists():
        steps += [f"uv pip install {py} -q -r {lock.resolve()} {pip_args}",
                  f"cd /content/mattergen && uv pip install {py} -q --no-deps -e ."]
    else:
        steps += [f"cd /content/mattergen && uv pip install {py} -q -e . {pip_args}",
                  f'uv pip install {py} -q "setuptools<81"']
    start_job("install", " && ".join(steps), "mg_install.log", shell=True, executable="/bin/bash")
    print("GPU found. MatterGen installation:", job_state("install"), "(log: mg_install.log)")
else:
    print("No GPU: sections D-F use the structures the TA generated with the same model, conditions and seed.")

## A. Molecule data

### A-1. Load the molecules

Each row is one QM9 molecule: its SMILES and its HOMO, LUMO and gap energies in eV, calculated with DFT. The only
property the model uses is the gap (LUMO − HOMO). We train on 8,000 of the 20,000 molecules, so that training takes
about a minute on a CPU. The cell also keeps the training SMILES in `train_smiles`: section C validates novelty
against this set: a generated molecule is new if it is not in it.

In [ ]:
qm9 = pd.read_csv(DATA_DIR / "qm9_subset_20k.csv")
N_TRAIN = 8000
train_df = qm9.sample(N_TRAIN, random_state=0).reset_index(drop=True)
## The graphs carry no stereochemistry, so molecules are compared by their SMILES without it.
flat_smiles = lambda mol: Chem.MolToSmiles(mol, isomericSmiles=False)
train_smiles = {flat_smiles(Chem.MolFromSmiles(s)) for s in train_df.smiles}
print(train_df.shape)
train_df.describe().round(2)

### A-2. Molecules as tensors

These are the functions from the preclass notebook (C-2 and C-3). Every molecule becomes two arrays of fixed size:

- **adjacency tensor** (5, 9, 9): for each pair of atom slots, one of 5 channels is 1: single, double, triple or
  aromatic bond, or "no bond";
- **feature matrix** (9, 5): for each of the 9 atom slots, one of 5 columns is 1: C, N, O, F, or "no atom".

Why a fixed size? The decoder (B-2) always outputs arrays of one shape, but QM9 molecules have up to 9 heavy atoms.
So a smaller molecule fills its unused slots with "no atom" (*padding*). With the "no" options, every slot and every
pair is a choice among exactly 5 classes: the decoder gives a probability for each class, and it makes a smaller
molecule by choosing "no atom" for some slots.

In [ ]:
ATOMS = ["C", "N", "O", "F"]                       ## heavy atoms in QM9 (hydrogens stay implicit)
BOND_TYPES = [BondType.SINGLE, BondType.DOUBLE, BondType.TRIPLE, BondType.AROMATIC]
NUM_ATOMS = 9                                      ## QM9 molecules have at most 9 heavy atoms
ATOM_DIM = len(ATOMS) + 1                          ## + one column for "no atom"
BOND_DIM = len(BOND_TYPES) + 1                     ## + one channel for "no bond"

def smiles_to_graph(smiles):
    """SMILES -> adjacency tensor (BOND_DIM, NUM_ATOMS, NUM_ATOMS) and feature matrix (NUM_ATOMS, ATOM_DIM)."""
    mol = Chem.MolFromSmiles(smiles)
    adjacency = np.zeros((BOND_DIM, NUM_ATOMS, NUM_ATOMS), "float32")
    features = np.zeros((NUM_ATOMS, ATOM_DIM), "float32")
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        features[i, ATOMS.index(atom.GetSymbol())] = 1
        for neighbor in atom.GetNeighbors():
            j = neighbor.GetIdx()
            b = BOND_TYPES.index(mol.GetBondBetweenAtoms(i, j).GetBondType())
            adjacency[b, [i, j], [j, i]] = 1
    adjacency[-1, adjacency.sum(axis=0) == 0] = 1   ## pairs without a bond use the "no bond" channel
    features[features.sum(axis=1) == 0, -1] = 1      ## empty rows use the "no atom" column
    return adjacency, features


adjacency, features = smiles_to_graph(train_df.smiles[0])
print(train_df.smiles[0], "| adjacency", adjacency.shape, "| features", features.shape)

`graph_to_molecule` goes the other way, from the two tensors back to an RDKit molecule. Every graph the decoder
produces passes through it in section C:

1. It drops the slots whose atom type is "no atom", and atoms without a bond to any other atom.
2. It adds one atom for each remaining slot, with the atom type in that slot's row of `features`.
3. It adds a bond for every pair (i, j) whose channel in `adjacency` is a bond type and not "no bond". It reads only
   the pairs with i < j, because the tensor is symmetric and each bond appears twice.
4. RDKit validates the chemistry (RDKit calls this step *sanitization*): no atom may have more bonds than it can
   make (C 4, N 3, O 2, F 1), and aromatic rings must be consistent. If this fails, the function returns `None`, and
   section C counts the sample as invalid.

Hydrogens are not in the graph: RDKit adds them afterwards to fill each atom's remaining bonds.

In [ ]:
def graph_to_molecule(adjacency, features):
    """Tensors -> RDKit molecule, or None if RDKit cannot sanitize it."""
    keep = np.where((features.argmax(axis=1) != ATOM_DIM - 1) & (adjacency[:-1].sum(axis=(0, 1)) != 0))[0]
    features, adjacency = features[keep], adjacency[:, keep][:, :, keep]
    mol = Chem.RWMol()
    for t in features.argmax(axis=1):
        mol.AddAtom(Chem.Atom(ATOMS[t]))
    for b, i, j in zip(*np.where(np.triu(adjacency) == 1)):
        if i == j or b == BOND_DIM - 1:
            continue
        mol.AddBond(int(i), int(j), BOND_TYPES[b])
    if Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
        return None
    return mol

Now we convert the 8,000 training molecules and the gap into PyTorch tensors. The gap is *standardised*: we subtract
its mean and divide by its standard deviation, so the values are centred at 0 and most lie between about −2 and +2.
Why standardise the target?

- Centring removes the average gap of about 7 eV, and scaling gives the target a size of about 1, a convenient size
  for a network whose weights start small. This can make training easier.
- The gap error is added to the other loss terms (B-3). Standardised, its size no longer depends on the unit (eV or
  hartree), so the property term cannot outweigh the others just because of the unit we chose.
- To turn a prediction back into eV, multiply it by `GAP_STD` and add `GAP_MEAN`.

In [ ]:
graphs = [smiles_to_graph(s) for s in train_df.smiles]
A_train = torch.tensor(np.stack([g[0] for g in graphs]))
X_train = torch.tensor(np.stack([g[1] for g in graphs]))
GAP_MEAN, GAP_STD = train_df.gap_eV.mean(), train_df.gap_eV.std()
P_train = torch.tensor(((train_df.gap_eV - GAP_MEAN) / GAP_STD).values, dtype=torch.float32)
print("adjacency", tuple(A_train.shape), "| features", tuple(X_train.shape), "| gap", tuple(P_train.shape))

## B. A graph VAE in PyTorch

A VAE has two halves.

- The **encoder** reads a molecule and returns a small region of a *latent space*: a mean `z_mean` and a
  log-variance `log_var` for each of the latent numbers. A point `z` is drawn from that region.
- The **decoder** turns `z` back into an adjacency tensor and a feature matrix.

Training asks for two things at once:

1. **Reconstruction:** the decoded graph should match the input.
2. **KL divergence:** the regions should stay close to a standard normal distribution.

To generate, we need only the decoder: a random `z` from a standard normal distribution goes through the decoder
and gives a new molecule (section C). Two parts are specific to our model:

- It reads graphs instead of SMILES strings, so the encoder starts with a graph convolution (B-1).
- A small extra layer, the *property head*, predicts the HOMO–LUMO gap from `z_mean`, which arranges the latent
  space by the gap (C-3).

The sampling formula is as follows, where ε is drawn from a standard normal distribution:

$$\mathbf{z} = \boldsymbol{\mu} + \exp\left(\tfrac{1}{2}\log\boldsymbol{\sigma}^2\right) \odot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, I)$$

Here μ is `z_mean`, log σ² is `log_var`, and ⊙ multiplies element by element.

| part | input | output |
|---|---|---|
| encoder (B-1, B-2) | adjacency (5, 9, 9) and features (9, 5) | `z_mean` and `log_var`, 435 numbers each |
| sampling (B-2) | `z_mean`, `log_var` | `z = z_mean + exp(log_var / 2) · ε`, with ε drawn from a standard normal distribution |
| decoder (B-2) | `z` | bond-type probabilities (5, 9, 9) and atom-type probabilities (9, 5) |
| property head (B-2) | `z_mean` | the predicted gap |
| loss (B-3) | all of the above | reconstruction + β · KL + property error + gradient penalty |

### B-1. Relational graph convolution

A graph convolution lets every atom collect information from the atoms bonded to it. The layer below does this in
three steps:

1. **Collect:** for each atom, add up the rows of `features` (the atom types) of its neighbours.
   `adjacency @ features` does this for all atoms at once.
2. **Keep bond types apart:** `adjacency` has one channel per bond type, so neighbours across single, double,
   triple and aromatic bonds are added up separately. This is what *relational* means.
3. **Weigh and combine:** each bond type has its own weight matrix (`kernel`); the weighted results are added up
   and passed through a ReLU.

Example: in ethanol (`CCO`), the middle carbon has two single-bond neighbours, a C and an O, so step 1 gives it
(C 1, N 0, O 1, F 0, no atom 0) in the single-bond channel. After the layer, each atom's new numbers describe which
atoms surround it and how they are bonded. The fifth channel, "no bond", collects the atoms that are *not* bonded to
it, so every atom also gets a summary of the rest of the molecule.

In [ ]:
class RelationalGraphConv(nn.Module):
    def __init__(self, in_dim, units):
        super().__init__()
        self.kernel = nn.Parameter(torch.empty(BOND_DIM, in_dim, units))
        nn.init.xavier_uniform_(self.kernel)

    def forward(self, adjacency, features):              ## (B, 5, 9, 9) and (B, 9, F)
        x = adjacency @ features.unsqueeze(1)              ## neighbours' features, per bond type
        x = x @ self.kernel                                ## one weight matrix per bond type
        return torch.relu(x.sum(dim=1))                    ## (B, 9, units)

### B-2. Encoder, decoder and VAE

- **Encoder:** one graph convolution (9 neurons), the mean over the atoms, a dense layer of 512 neurons, and two outputs
  of size `LATENT_DIM = 435`: the mean `z_mean` and the log-variance `log_var`.
- **Decoder:** dense layers of 128, 256 and 512 neurons with 20 % dropout, then two outputs reshaped into an adjacency
  tensor and a feature matrix. A softmax turns each entry into probabilities over the bond types or atom types, and
  the adjacency is made symmetric, because a bond from i to j is also a bond from j to i.
- **Property head:** a linear layer predicts the (standardised) gap from `z_mean`. It shapes the latent space so that
  molecules with similar gaps sit close together.

In [ ]:
LATENT_DIM = 435

class Encoder(nn.Module):
    def __init__(self, latent_dim, gconv_units=(9,), dense_units=(512,), dropout=0.0):
        super().__init__()
        dims = [ATOM_DIM, *gconv_units]
        self.gconvs = nn.ModuleList(RelationalGraphConv(a, b) for a, b in zip(dims[:-1], dims[1:]))
        layers, d = [], dims[-1]
        for u in dense_units:
            layers += [nn.Linear(d, u), nn.ReLU(), nn.Dropout(dropout)]
            d = u
        self.dense = nn.Sequential(*layers)
        self.z_mean, self.log_var = nn.Linear(d, latent_dim), nn.Linear(d, latent_dim)

    def forward(self, adjacency, features):
        x = features
        for g in self.gconvs:
            x = g(adjacency, x)
        x = self.dense(x.mean(dim=1))                       ## average over the atoms
        return self.z_mean(x), self.log_var(x)

print(Encoder(LATENT_DIM))

The decoder and the full VAE. In the printout, the decoder's two output layers (`adj` and `feat`) have 405 = 5 × 9 × 9 and 45 = 9 × 5 outputs: they are reshaped into the adjacency tensor and the feature matrix.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, dense_units=(128, 256, 512), dropout=0.2):
        super().__init__()
        layers, d = [], latent_dim
        for u in dense_units:
            layers += [nn.Linear(d, u), nn.Tanh(), nn.Dropout(dropout)]
            d = u
        self.dense = nn.Sequential(*layers)
        self.adj = nn.Linear(d, BOND_DIM * NUM_ATOMS * NUM_ATOMS)
        self.feat = nn.Linear(d, NUM_ATOMS * ATOM_DIM)

    def forward(self, z):
        x = self.dense(z)
        a = self.adj(x).view(-1, BOND_DIM, NUM_ATOMS, NUM_ATOMS)
        a = (a + a.transpose(2, 3)) / 2                     ## symmetric: bond i-j = bond j-i
        f = self.feat(x).view(-1, NUM_ATOMS, ATOM_DIM)
        return a.softmax(dim=1), f.softmax(dim=2)

class MoleculeVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder, self.decoder = Encoder(latent_dim), Decoder(latent_dim)
        self.property_head = nn.Linear(latent_dim, 1)

    def forward(self, adjacency, features):
        z_mean, log_var = self.encoder(adjacency, features)
        z = z_mean + torch.exp(0.5 * log_var) * torch.randn_like(z_mean)    ## reparameterisation trick
        adj_gen, feat_gen = self.decoder(z)
        return z_mean, log_var, self.property_head(z_mean).squeeze(1), adj_gen, feat_gen

print(Decoder(LATENT_DIM))
print(sum(p.numel() for p in MoleculeVAE().parameters()), "parameters in the whole VAE")

### B-3. The loss

The loss has five terms:

1. **adjacency:** cross-entropy between the true and the decoded bond types of every atom pair;
2. **features:** cross-entropy between the true and the decoded atom type of every slot;
3. **KL divergence**, multiplied by `beta`: keeps the latent space close to a standard normal distribution, so random
   points decode to molecules. We use `BETA = 0.01`: with `beta = 1` the KL term wins, the decoder learns to ignore `z`, and almost every sample is the same molecule (*posterior collapse*).
   Practice 1 lets you see this;
4. **property:** mean squared error of the predicted gap;
5. **gradient penalty:** asks the gradient of the property head with respect to the input graph to have size about 1,
   on graphs mixed from a real and a decoded one. It keeps the property head smooth.

In [ ]:
def gradient_penalty(model, adjacency, features, adj_gen, feat_gen):
    alpha = torch.rand(len(adjacency), 1, 1, 1)
    adj_mix = (alpha * adjacency + (1 - alpha) * adj_gen.detach()).requires_grad_(True)
    feat_mix = (alpha[..., 0] * features + (1 - alpha[..., 0]) * feat_gen.detach()).requires_grad_(True)
    prop = model(adj_mix, feat_mix)[2]
    g_adj, g_feat = torch.autograd.grad(prop.sum(), [adj_mix, feat_mix], create_graph=True)
    return ((1 - g_adj.norm(dim=1)) ** 2).mean(dim=(1, 2)).add(((1 - g_feat.norm(dim=2)) ** 2).mean(dim=1)).mean()

def vae_loss(model, adjacency, features, prop, beta=1.0):
    z_mean, log_var, prop_pred, adj_gen, feat_gen = model(adjacency, features)
    eps = 1e-7
    adjacency_loss = -(adjacency * torch.log(adj_gen + eps)).sum(dim=1).sum(dim=(1, 2)).mean()
    features_loss = -(features * torch.log(feat_gen + eps)).sum(dim=2).sum(dim=1).mean()
    kl = (-0.5 * (1 + log_var - z_mean**2 - log_var.exp().clamp(max=1e6)).sum(dim=1)).mean()
    property_loss = F.mse_loss(prop_pred, prop)
    penalty = gradient_penalty(model, adjacency, features, adj_gen, feat_gen)
    return adjacency_loss + features_loss + beta * kl + property_loss + penalty

### B-4. Train on a CPU

`train_vae` builds a new model and trains it with Adam (learning rate 5e-4, batches of 100). The loss
should fall over the epochs. This takes about one minute on a Colab CPU; Practice 1 calls the same function with
another `beta`.

In [ ]:
EPOCHS, BETA = 20, 0.01

def train_vae(beta=BETA, latent_dim=LATENT_DIM, epochs=EPOCHS, seed=0, verbose=True):
    torch.manual_seed(seed)
    model = MoleculeVAE(latent_dim)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4)
    loader = DataLoader(TensorDataset(A_train, X_train, P_train), batch_size=100, shuffle=True)
    history = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for a, x, p in loader:
            loss = vae_loss(model, a, x, p, beta)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(a)
        history.append(total / len(A_train))
        if verbose and (epoch % 5 == 4 or epoch == 0):
            print(f"epoch {epoch + 1:2d} | loss {history[-1]:.2f}")
    return model, history

t0 = time.time()
vae, history = train_vae()
print(f"trained in {time.time() - t0:.0f} s")
plt.figure(figsize=(10, 2.2)); plt.plot(range(1, EPOCHS + 1), history, marker="o"); plt.tick_params(labelsize=12)
plt.xlabel("epoch", fontsize=13); plt.ylabel("training loss", fontsize=13); plt.title("Graph VAE training", fontsize=13)
plt.show()

## C. Generate and validate molecules

### C-1. Sample from the latent space

New molecules come from random points `z` drawn from a standard normal distribution. The decoder turns each point
into probabilities; we keep the most likely bond type for every pair and the most likely atom type for every slot,
remove bonds from an atom to itself, and rebuild the molecule.

In [ ]:
@torch.no_grad()
def sample_molecules(model, n=1000, seed=1):
    model.eval()
    z = torch.randn(n, model.latent_dim, generator=torch.Generator().manual_seed(seed))
    adj, feat = model.decoder(z)
    adj = F.one_hot(adj.argmax(dim=1), BOND_DIM).permute(0, 3, 1, 2).float() * (1 - torch.eye(NUM_ATOMS))
    feat = F.one_hot(feat.argmax(dim=2), ATOM_DIM).float()
    return [graph_to_molecule(a.numpy(), f.numpy()) for a, f in zip(adj, feat)]

mols = sample_molecules(vae)
print(len(mols), "sampled;", sum(m is not None for m in mols), "passed the RDKit chemistry validation")

### C-2. Three validation metrics: validity, uniqueness, novelty

- **Validity:** the share of samples that RDKit accepts as one connected molecule. This is stricter than the count
  printed in C-1: a sample with no atoms left, or made of two or more separate pieces (its SMILES contains a `.`),
  passes RDKit but is not valid here.
- **Uniqueness:** the share of the valid molecules that are different from each other.
- **Novelty:** the share of the unique molecules that are not in the training set (`train_smiles`, A-1). It is
  measured against our 8,000 training molecules only, so a "new" molecule may still be in QM9 or another database.

A model can score high on one and low on another: copying training molecules gives perfect validity and zero
novelty. The figure shows unique valid molecules with their QED drug-likeness score.

In [ ]:
def molecule_metrics(mols, train_smiles):
    smiles = [flat_smiles(m) for m in mols if m is not None and m.GetNumAtoms() > 0]
    smiles = [s for s in smiles if "." not in s]                      ## one connected molecule only
    unique = set(smiles)
    return {"n_sampled": len(mols), "validity": len(smiles) / len(mols),
            "uniqueness": len(unique) / max(len(smiles), 1),
            "novelty": len(unique - train_smiles) / max(len(unique), 1)}

def valid_unique(mols):
    out, seen = [], set()
    for m in mols:
        if m is not None and m.GetNumAtoms() > 0:
            s = flat_smiles(m)
            if "." not in s and s not in seen:
                seen.add(s); out.append(m)
    return out

metrics = molecule_metrics(mols, train_smiles)
print({k: round(v, 3) for k, v in metrics.items()})
good = valid_unique(mols)
print(len(good), "valid, unique molecules")
opts = Draw.MolDrawOptions(); opts.legendFontSize, opts.minFontSize = 22, 16      ## readable labels
Draw.MolsToGridImage(good[:9], molsPerRow=3, subImgSize=(240, 200), drawOptions=opts,
                     legends=[f"QED {QED.qed(m):.2f}" for m in good[:9]])

### C-3. Do the new molecules resemble the training molecules?

The first plot compares the number of heavy atoms; most training molecules have 9. A generated molecule is smaller
when the decoder chose "no atom" for some slots or left atoms without a bond, which `graph_to_molecule` drops (A-2). The second shows the training molecules in the latent space,
reduced to two dimensions with PCA and coloured by their gap. If the colour changes smoothly from one side of the
cloud to the other, from low gaps to high gaps, the property head has organised the latent space by the gap, which is what lets a VAE search for molecules with a target property.

In [ ]:
heavy_gen = [m.GetNumHeavyAtoms() for m in good]
heavy_train = [Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in train_df.smiles[:2000]]
with torch.no_grad():
    z_mean = vae.encoder(A_train[:2000], X_train[:2000])[0].numpy()
z2 = PCA(n_components=2, random_state=0).fit_transform(z_mean)       ## random_state: the same picture every run
bins = np.arange(0.5, 10.5)
plt.figure(figsize=(5, 3.4))
plt.hist(heavy_train, bins=bins, density=True, alpha=0.6, label="training")
plt.hist(heavy_gen, bins=bins, density=True, alpha=0.6, label="generated")
plt.xlabel("heavy atoms"); plt.ylabel("share"); plt.legend(); plt.title("Size of the molecules"); plt.show()
plt.figure(figsize=(5.4, 4.6))
sc = plt.scatter(z2[:, 0], z2[:, 1], c=train_df.gap_eV[:2000], s=4, cmap="viridis")
plt.colorbar(sc, label="gap (eV)"); plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("Training molecules in the latent space"); plt.show()

### C-4. Practice 1

The KL weight `beta` trades two goals against each other. A small `beta` lets the encoder spread the molecules out,
which gives variety but leaves gaps in the latent space where the decoder writes invalid graphs; a large `beta` packs
the space tightly, which gives valid but repetitive molecules.

Train a second model with `train_vae(beta=...)`, for example `beta=1` or `beta=0.1`, sample 1,000
molecules with `sample_molecules`, and compare `molecule_metrics` with C-2 (`beta = 0.01`). Which of the three metrics
changes most, and in which direction?

Read the three shares together. Uniqueness is a share of the valid molecules and novelty a share of the unique
ones, so validity × uniqueness × 1,000 is the number of different valid molecules. A novelty of 1 among a handful of
unique molecules says little.

In [ ]:
## Write your code here.

## D. MatterGen: a diffusion model for crystals

In C-1, the VAE from B made a molecule in one step: one random `z`, one pass through the decoder. MatterGen is a
**diffusion model**: it builds a crystal in many small steps (see the slide: Fig. 1 of the MatterGen paper).

- **Training:** a *forward process* corrupts real crystals step by step until only noise is left: more and more atom
  types become a "masked" placeholder, the fractional coordinates get random shifts that wrap around the cell
  border, and the lattice drifts towards a random cell around a cubic one. A *score network* learns to undo one
  step: it predicts how to make the atom types, coordinates and lattice less noisy.
- **Generation:** start from pure noise (every atom masked, random positions, a random cell) and remove a little
  noise in each of 1,000 steps.

The VAE needs one network pass; MatterGen needs at least one per step, so it runs on a GPU and takes minutes.

MatterGen was trained on about 600,000 stable structures with at most 20 atoms per cell (the Alex-MP-20 data set),
then *fine-tuned* with *adapter* modules, added to each layer, that feed in property labels. The model we use,
`chemical_system_energy_above_hull`, was fine-tuned on two **conditions**:

- `chemical_system`, the elements the crystal should contain, for example `Li-P-S`;
- `energy_above_hull`, the target energy above the convex hull (0.05 eV/atom here, see D-1).

During fine-tuning the labels were sometimes hidden, so the network can predict each step with and without the
conditions. **Classifier-free guidance** mixes the two predictions with the **guidance factor** γ:

`prediction = (1 − γ) · unconditional + γ · conditional`

γ = 0 ignores the conditions (our baseline C1); γ = 1 uses the conditional prediction alone; γ > 1 pushes beyond it
and follows the conditions more strictly, often at the cost of variety.

Our target is **sulfide solid electrolytes**: Li-ion conductors such as Li3PS4, Li7P3S11 and the argyrodite
Li6PS5Cl, which can replace the liquid electrolyte in a battery.

### D-1. The conditions and your choice

The TA generated structures for these conditions on a Colab T4 with the same model and seed 42: 16 per condition
in one run, of which each set in `Data/` keeps the first 8 (`gen_0` to `gen_7`); a run of
your own in D-2 makes 8 new ones. The cell prints which sets are in `Data/` and how many structures each
holds. C1 is the baseline. Choose one of the other conditions now; section F uses it. E-2 already shows C3 against C1, so another condition gives you a new comparison.

| condition | chemical system | energy above hull (eV/atom) | guidance γ | role |
|---|---|---|---|---|
| C1 | Li-P-S | 0.05 | 0.0 | baseline: guidance 0 = unconditional |
| C2 | Li-P-S | 0.05 | 1.0 | student choice |
| C3 | Li-P-S | 0.05 | 2.0 | student choice |
| C4 | Li-P-S | 0.05 | 4.0 | student choice |
| C5 | Li-P-S-Cl | 0.05 | 2.0 | student choice |
| C6 | Li-P-S-Br | 0.05 | 2.0 | student choice |

**Why these conditions?** C1 to C4 differ only in γ (the effect of guidance); C5 and C6 add a halogen, as in the
argyrodite Li6PS5Cl.

**Energy above hull:** plot the energy per atom (from DFT) of all known phases of a chemical system against
composition; the *convex hull* runs through the lowest points, with none below it. A phase above the hull can lower
its energy by decomposing into hull phases, so 0.05 eV/atom means "nearly stable". MatterGen only aims for this
value; the advanced notebook validates it.

In [ ]:
CONDITIONS = {
    "C1": {"chemical_system": "Li-P-S", "energy_above_hull": 0.05, "guidance": 0.0},
    "C2": {"chemical_system": "Li-P-S", "energy_above_hull": 0.05, "guidance": 1.0},
    "C3": {"chemical_system": "Li-P-S", "energy_above_hull": 0.05, "guidance": 2.0},
    "C4": {"chemical_system": "Li-P-S", "energy_above_hull": 0.05, "guidance": 4.0},
    "C5": {"chemical_system": "Li-P-S-Cl", "energy_above_hull": 0.05, "guidance": 2.0},
    "C6": {"chemical_system": "Li-P-S-Br", "energy_above_hull": 0.05, "guidance": 2.0},
}
STUDENT_MENU = ['C2', 'C3', 'C4', 'C5', 'C6']

## Choose your condition: one of C2, C3, C4, C5, C6.
MY_CONDITION = "C3"

assert MY_CONDITION in STUDENT_MENU, f"Choose one of {STUDENT_MENU}"
TA_SETS = json.loads((DATA_DIR / "mattergen_pregenerated_provenance.json").read_text())["sets"]
print("TA sets:", {k: v["n_structures"] for k, v in TA_SETS.items()})
if MY_CONDITION not in TA_SETS and not globals().get("USE_GPU", False):
    raise ValueError(f"The TA's set for {MY_CONDITION} is not in Data/ yet; choose one of {sorted(set(TA_SETS) & set(STUDENT_MENU))}.")
print("Your condition:", MY_CONDITION, CONDITIONS[MY_CONDITION])

### D-2. On a GPU runtime: generate your condition in the background

If MatterGen has finished installing, this cell starts generating 8 structures for your condition and returns
at once. Plan for up to about 15 minutes: on the TA's T4, runs of 16 took about 8 minutes for C1 and C2 and about 14
minutes for C3-C6 (with guidance above 1, every step evaluates both the conditional and the unconditional
prediction); a run of 8 has not been timed yet and
should not take longer. If the installation is still running, run this cell again in a minute; running it again never
starts a second generation. Without a GPU, or if anything fails, section F uses the TA's
structures for the same condition, so you never wait.

**What runs:** `mattergen-generate`, with your condition, seed 42 and 8 structures, writing
`generate.log` and the structures into `mg_live/<condition>/`. **What to check:** the printed state. `running`: go
on with D-3 and E; `done`: F-2 will use your structures; `failed`: F-2 uses the TA's set.

In [ ]:
if not globals().get("USE_GPU", False):
    print("No GPU: section F will use the TA's structures for", MY_CONDITION)
elif job_state("install") == "running":
    print("MatterGen is still installing. Run this cell again in a minute.")
elif job_state("install") != "done":
    print("The installation", job_state("install"), "- section F will use the TA's structures.")
else:
    c = CONDITIONS[MY_CONDITION]
    live_dir = Path("mg_live") / MY_CONDITION
    busy = [n for n in JOBS if n.startswith("generate-") and n != "generate-" + MY_CONDITION and job_state(n) == "running"]
    if busy:
        print("Another generation is still running:", busy, "- wait for it, then run this cell again.")
    else:
        props = {"chemical_system": c["chemical_system"], "energy_above_hull": c["energy_above_hull"]}
        cmd = [MG + "/mattergen-generate", str(live_dir), f"--pretrained-name={MODEL}", f"--batch_size={BATCH_SIZE}",
               "--num_batches=1", f"--properties_to_condition_on={props!r}",
               f"--diffusion_guidance_factor={c['guidance']}", f"--seed={SEED}", "--record_trajectories=False"]
        if job_state("generate-" + MY_CONDITION) not in ("running", "done"):
            live_dir.mkdir(parents=True, exist_ok=True)       ## the request travels with the output
            (live_dir / "request.json").write_text(json.dumps({"condition": MY_CONDITION, **c, "model": MODEL,
                                                               "seed": SEED, "batch_size": BATCH_SIZE}))
        env = dict(os.environ, PATH=MG + ":" + os.environ["PATH"], HF_HOME="/content/.hf_cache")
        start_job("generate-" + MY_CONDITION, cmd, str(live_dir / "generate.log"), env=env)
        print(MY_CONDITION, "generation:", job_state("generate-" + MY_CONDITION), f"(log: {live_dir}/generate.log)")

### D-3. The TA's structures: baseline C1 and conditioned C3

While your own run (if any) works in the background, we look at two of the TA's sets: the first 8 of the 16
structures that one T4 run made for C1 and for C3 (see D-1). Each zip holds one CIF file per structure. The table
lists the formula, the number of atoms in the cell and the space group.

- **formula:** C1 (γ = 0) ignores the target, so any elements can appear; C3 should have only Li, P and S.
- **atoms:** per unit cell; the training structures have at most 20.
- **space group:** found with a 0.1 Å tolerance (`symprec=0.1`); P1 means no symmetry beyond the repeat of the
  cell, P4/mmm or F-43m much more.

Pictures: dashed lines are cell edges; Li purple, P orange, S yellow.

In [ ]:
def load_cifs(zip_path):
    """Read every CIF in a MatterGen output zip as a pymatgen Structure."""
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".cif"))
        return [Structure.from_str(zf.read(n).decode(), fmt="cif") for n in names]


baseline_structures = load_cifs(PREGEN / "C1" / "generated_crystals_cif.zip")
c3_structures = load_cifs(PREGEN / "C3" / "generated_crystals_cif.zip")
overview = pd.DataFrame([{"set": name, "formula": s.composition.reduced_formula, "atoms": len(s),
                          "space group": s.get_space_group_info(symprec=0.1)[0]}
                         for name, group in (("C1", baseline_structures), ("C3", c3_structures)) for s in group])
print(len(baseline_structures), "baseline and", len(c3_structures), "C3 structures |",
      "made on:", {k: TA_SETS[k]["device"] for k in ("C1", "C3")})
overview.head(12)

In [ ]:
n_show = min(4, len(c3_structures))
ncols = min(2, n_show)
fig, axes = plt.subplots(math.ceil(n_show / ncols), ncols, figsize=(4 * ncols, 4 * math.ceil(n_show / ncols)),
                         squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, s in zip(axes.flat, c3_structures[:n_show]):
    plot_atoms(AseAtomsAdaptor.get_atoms(s), ax, radii=0.5, rotation="15x,30y,0z")
    ax.set_title(f"{s.composition.reduced_formula} ({len(s)} atoms)", fontsize=13)
fig.suptitle(f"{n_show} C3 structures (Li-P-S, guidance 2)", fontsize=14)
plt.show()

## E. Validate the crystals

### E-1. Four validation criteria

1. **Target hit:** does the crystal contain exactly the requested elements?
2. **Distance:** is the shortest distance from any atom to another atom at least 0.5 Å? The search includes the
   periodic copies in neighbouring cells, also the copies of the same atom. Closer atoms are unphysical.
3. **Charge balance:** can the atoms take oxidation states that pymatgen lists for each element from known crystals
   (Li +1; P from −3 to +5; S from −2 to +6; Cl −1; …) so that they add up to zero? Atoms of one element may take
   different states (mixed valence), so this is a loose test: LiP2S passes with P at +0.5 on average.
4. **Known phase:** does the structure match a known Materials Project crystal (energy above hull ≤ 0.1 eV/atom)
   after allowing for small distortions? A match means the model rediscovered a known material; no match means it
   generated something new, which still needs to be validated for stability (see the advanced notebook). For these Li–P–S
   structures the share of known phases is often 0: a valid result, since the model mostly generates compositions that
   are not in the reference file.

What a failure means. No **target hit**: the model missed the condition, though the crystal may be sensible.
**Distance** below 0.5 Å: two atoms nearly coincide, a broken structure. No **charge balance**: a Li–P–S compound
cannot be split into ions. Crystals made only of metals often fail too: metals bond without forming ions, so for them
a failure does not mean *impossible*.

In [ ]:
refs = json.load(gzip.open(DATA_DIR / "mp_LiPS_references.json.gz", "rt"))
REF_STRUCTURES = {}
for e in refs:
    REF_STRUCTURES.setdefault(Composition(e["formula_pretty"]).reduced_formula, []).append(Structure.from_dict(e["structure"]))
matcher = StructureMatcher()
print(len(refs), "known phases with", len(REF_STRUCTURES), "formulas")

def min_distance(s):
    """Shortest distance from an atom to another atom or to a periodic copy, also a copy of itself."""
    r = min(s.lattice.abc) + 0.01          ## each atom has a copy one lattice length away, so the search finds one
    return float(s.get_neighbor_list(r)[3].min())

def charge_balanced(s):
    return len(s.composition.reduced_composition.oxi_state_guesses()) > 0

def check_structure(s, system):
    formula = s.composition.reduced_formula
    known = REF_STRUCTURES.get(formula, [])
    d_min = min_distance(s)
    return {"formula": formula, "atoms": len(s), "space_group": s.get_space_group_info(symprec=0.1)[0],
            "target_hit": {el.symbol for el in s.composition.elements} == set(system.split("-")),
            "min_distance_A": round(d_min, 3),
            "valid_distance": d_min >= 0.5,
            "charge_balanced": charge_balanced(s),
            "known_structure": any(matcher.fit(s, r) for r in known)}

def structures_digest(structures):
    """A fingerprint of the structures themselves (lattice, species, positions), not only of their formulas."""
    return hashlib.sha256(json.dumps([s.as_dict() for s in structures], sort_keys=True).encode()).hexdigest()[:16]

def check_set(structures, system):
    return pd.DataFrame([check_structure(s, system) for s in structures])

def rates(table):
    return {"hit_rate": table.target_hit.mean(), "valid_rate": table.valid_distance.mean(),
            "charge_balanced_rate": table.charge_balanced.mean(), "known_match_rate": table.known_structure.mean()}

c3_table = check_set(c3_structures, CONDITIONS["C3"]["chemical_system"])
c3_table

### E-2. Baseline C1 against conditioned C3

The bars show the share of structures that pass each criterion; the error bars are 95 % Wilson intervals. With n
structures per set, one structure changes a share by 1/n (12.5 percentage points for 8),
so small differences are noise. Both sets are the first 8 structures of one T4 run each (D-1).

A **Wilson interval** is the range of true shares consistent with the count (95 % confidence). With 8 structures,
0 of 8 gives 0 to 0.32 and 8 of 8 gives 0.68 to 1. A bar of height 0 shows only its error bar. A rough visual rule:
if two intervals do not overlap, the shares differ clearly; if they overlap, the plot alone cannot settle it (a
statistical test may still find a difference), and it never shows that the shares are equal.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p, d = k / n, 1 + z * z / n
    centre, half = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half

def plot_rates(tables):
    """Bar chart of the four shares for each named table, with 95 % Wilson intervals."""
    names = ["hit_rate", "valid_rate", "charge_balanced_rate", "known_match_rate"]
    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    width = 0.8 / len(tables)
    for k, (label, t) in enumerate(tables.items()):
        r = rates(t)
        ci = [wilson(round(r[n] * len(t)), len(t)) for n in names]
        err = np.array([[r[n] - lo for n, (lo, hi) in zip(names, ci)], [hi - r[n] for n, (lo, hi) in zip(names, ci)]])
        ax.bar(np.arange(4) + (k - (len(tables) - 1) / 2) * width, [r[n] for n in names], width,
               yerr=err.clip(min=0), capsize=3, label=f"{label} ({len(t)})")
    ax.set_xticks(range(4), ["target\nhit", "distance\n≥ 0.5 Å", "charge\nbalance", "known\nphase"])
    ax.set_ylim(0, 1.3); ax.set_yticks(np.linspace(0, 1, 6)); ax.tick_params(labelsize=13)
    ax.set_ylabel("share of structures", fontsize=13)
    ax.legend(loc="upper left", ncol=len(tables), fontsize=12)    ## above the bars, which stop at 1
    return fig

baseline_table = check_set(baseline_structures, CONDITIONS["C1"]["chemical_system"])
print(pd.DataFrame({"C1": rates(baseline_table), "C3": rates(c3_table)}).round(2))
plot_rates({"C1 baseline": baseline_table, "C3": c3_table}); plt.show()

### E-3. Practice 2

Pick one C3 structure that passes the first three criteria (target hit, distance and charge balance); if none passes
all three, take the one that passes the most. Report its formula, space group and density, the shortest Li–S
distance, and whether it matches a known phase. Hint: `c3_table` and `c3_structures` share the same order, and
`Structure.density` and `Structure.distance_matrix` give the numbers you need.

What to look for: is the Li–S distance close to the one in Li2S (2.455 Å, preclass D-3)?

In [ ]:
## Write your code here.

## F. Assignment (attendance): does the condition change what MatterGen makes?

You compare your condition from D-1 with the baseline C1 on the four validation criteria from E. Your structures come from your
own run if it finished on a GPU, otherwise from the TA's set for the same condition; both use the same model,
condition and seed, but need not give the same structures: the TA's set keeps the first 8 of a run of 16, and your
run makes 8 new ones (D-1). Fill in F-3 to F-5 with the tools from section E.

### F-1. Set your student ID

Replace `"demo"` with your student ID. `STUDENT_ID` names your submission files.

In [ ]:
## Replace this with your own student ID; it only names your files.
STUDENT_ID = "demo"

if STUDENT_ID == "demo":
    raise ValueError("Set STUDENT_ID to your own ID before running this cell.")
print("Student ID:", STUDENT_ID)

### F-2. Your structures

This cell uses your own structures from D-2 if your run for this condition finished, and the TA's otherwise.
`MY_SOURCE` records which one (`generated` or `pregenerated`); either is fine. If your run is still busy, go on
with the TA's set or run this cell again later. This cell stops a run still busy after 20 minutes.
Running this cell again clears the results of F-3 to F-6, so run them again afterwards.

In [ ]:
for name in ("my_table", "my_rates", "my_fig", "my_results"):   ## results of an earlier selection no longer apply
    globals().pop(name, None)
MY_SOURCE = "pregenerated"
live_dir = Path("mg_live") / MY_CONDITION
job = globals().get("JOBS", {}).get("generate-" + MY_CONDITION)
if job is not None:
    request_file = live_dir / "request.json"
    request = json.loads(request_file.read_text()) if request_file.exists() else {}
    expected = {"condition": MY_CONDITION, **CONDITIONS[MY_CONDITION], "model": MODEL, "seed": SEED,
                "batch_size": BATCH_SIZE}
    if job["proc"].poll() is None and time.time() - job["started"] > 1200:
        job["proc"].kill(); job["proc"].wait()
        print("Your run took longer than 20 minutes and was stopped.")
    elif job["proc"].poll() is None:
        print("Your run is still busy; run this cell again later to use it.")
    elif job["proc"].returncode == 0 and request == expected and (live_dir / "generated_crystals_cif.zip").exists():
        MY_SOURCE = "generated"
    else:
        print(f"Your run did not finish with the current settings (see {job['log']}).")
my_zip = (live_dir if MY_SOURCE == "generated" else PREGEN / MY_CONDITION) / "generated_crystals_cif.zip"
if not my_zip.exists():
    raise FileNotFoundError(f"No structures for {MY_CONDITION} yet: choose a condition with a TA set (see D-1).")
my_structures = load_cifs(my_zip)
MY_SELECTION = {"condition": MY_CONDITION, "source": MY_SOURCE, "zip": str(my_zip),
                "digest": structures_digest(my_structures)}
print(f"{MY_CONDITION}: {len(my_structures)} structures, source = {MY_SOURCE}")

### F-3. Validate every structure

Use `check_set` from E-1 to make `my_table`, one row per structure, validated against the chemical system of `MY_CONDITION`.

In [ ]:
## Write your code here.

### F-4. The four shares

Use `rates` from E-1 to make `my_rates`, a dictionary with the four shares.

In [ ]:
## Write your code here.

### F-5. The summary plot

Use `plot_rates` from E-2 to make `my_fig`, a bar chart of the baseline C1 (`baseline_table`) next to your condition (`my_table`), labelled with `MY_CONDITION`.

In [ ]:
## Write your code here.

### F-6. Results

This cell checks that F-2 to F-5 ran for your current condition and builds `my_results`, one row with your
condition, the source of the structures and the four shares. Compare them with the baseline C1 in your F-5 plot: did your
condition change the target hit or the share of known phases? Use the visual rule from E-2: error bars that do not
overlap show a clear change; with overlap, report "no clear change by this rule".

In [ ]:
def check_assignment():
    """Stop if the table, shares or plot do not belong to the structures that F-2 loaded for MY_CONDITION."""
    missing = [n for n in ("MY_SELECTION", "my_table", "my_rates", "my_fig") if n not in globals()]
    if missing:
        raise NameError(f"Complete F-2 to F-5 first. Not defined yet: {missing}")
    problems = []
    if MY_SELECTION["condition"] != MY_CONDITION:
        problems.append(f"MY_CONDITION is {MY_CONDITION}, but F-2 loaded {MY_SELECTION['condition']}: run F-2 again.")
    if MY_SELECTION["digest"] != structures_digest(my_structures):
        problems.append("my_structures changed after F-2: run F-2 again.")
    fresh = checked(my_structures, CONDITIONS[MY_CONDITION]["chemical_system"])       ## checked again, row by row
    if not my_table.reset_index(drop=True).equals(fresh):
        problems.append("my_table is not check_set(my_structures, ...) for your condition: run F-3 again.")
    want = rates(fresh)
    if set(my_rates) != set(want) or not all(np.isfinite(float(my_rates[k])) and np.isclose(float(my_rates[k]), v)
                                             for k, v in want.items()):
        problems.append("my_rates is not rates(my_table): run F-4 again.")
    base = checked(baseline_structures, CONDITIONS["C1"]["chemical_system"])
    if not (plot_shows(my_fig, fresh, MY_CONDITION) and plot_shows(my_fig, base)):
        problems.append("my_fig does not show the shares of my_table (labelled with MY_CONDITION) and of the "
                        "baseline: run F-5 again.")
    if problems:
        raise ValueError("\n".join(problems))

CHECKED = globals().get("CHECKED", {})
def checked(structures, system):
    """check_set, computed once per set of structures (F-7 reuses the result of F-6)."""
    key = (structures_digest(structures), system)
    if key not in CHECKED:
        CHECKED[key] = check_set(structures, system)
    return CHECKED[key]

def plot_shows(fig, table, label=""):
    """True if fig has bars labelled '<label>… (<n>)' whose heights are the table's four shares."""
    want = list(rates(table).values())
    return any(isinstance(c, BarContainer) and c.get_label().startswith(label)
               and c.get_label().endswith(f"({len(table)})")
               and np.allclose([p.get_height() for p in c.patches], want) for ax in fig.axes for c in ax.containers)

def results_row():
    """The submission row, with the columns always in the same order."""
    return pd.DataFrame([{"student_id": STUDENT_ID, "condition": MY_CONDITION, "source": MY_SELECTION["source"],
                          "n_structures": len(my_table), **{k: round(float(my_rates[k]), 4) for k in rates(my_table)}}])

check_assignment()
my_results = results_row()
print(my_results.T.to_string(header=False))           ## one row, printed as a column for reading

### F-7. Save your submission

Run the cell to save:

1. `hos8_<id>_results.csv`: your results row from F-6.
2. `hos8_<id>_summary.png`: your plot from F-5.

In Colab, the cell downloads both files. Allow both downloads if asked. Otherwise, find them in the `outputs` folder.

Upload both files to **KLMS (W14 Lab)**. Check the values and the plot yourself before submitting.

In [ ]:
## Stop with a clear message if F-6 has not been run, or if anything changed since.
if "my_results" not in globals():
    raise NameError("Complete F-6 first. Not defined yet: my_results")
check_assignment()
if not my_results.equals(results_row()):
    raise ValueError("my_results is out of date: run F-6 again.")
safe_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-")
stem = OUTPUT / f"hos8_{safe_id}"
paths = [Path(f"{stem}_results.csv"), Path(f"{stem}_summary.png")]
my_results.to_csv(paths[0], index=False)
my_fig.savefig(paths[1], dpi=180, bbox_inches="tight")
for path in paths:
    print("created" if path.stat().st_size > 0 else "MISSING", path)
try:   ## in Colab, download both files to your computer for the KLMS upload
    from google.colab import files
    files.download(str(paths[0])); files.download(str(paths[1]))
except ImportError:
    print("Not in Colab: download the two files from the outputs folder.")
print("Upload both files to KLMS (W14 Lab).")

## Data sources

- **QM9:** R. Ramakrishnan, P. O. Dral, M. Rupp, O. A. von Lilienfeld, *Scientific Data* 1, 140022 (2014),
  https://doi.org/10.1038/sdata.2014.22. Data: figshare, https://doi.org/10.6084/m9.figshare.c.978904, licensed under
  CC0 1.0 (https://creativecommons.org/publicdomain/zero/1.0/). `Data/qm9_subset_20k.csv` is a random sample of
  20,000 molecules (uncharacterised and charged molecules removed); see `Data/qm9_SOURCE.txt`.
- **Materials Project:** `Data/mp_LiPS_references.json.gz`, Li–P–S(–Cl, –Br) phases with energy above hull
  ≤ 0.1 eV/atom, accessed 2026-11-04, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/).
  Cite: A. Jain et al., *APL Materials* 1, 011002 (2013), https://doi.org/10.1063/1.4812323.
- **MatterGen:** C. Zeni et al., "A generative model for inorganic materials design", *Nature* 639, 624–632 (2025),
  https://doi.org/10.1038/s41586-025-08628-5. Code and checkpoints: https://github.com/microsoft/mattergen (MIT
  License). `Data/mattergen_pregenerated/` holds structures generated by the TA with this model; see
  `Data/mattergen_pregenerated_provenance.json`.
- The graph VAE follows the Keras example "Drug Molecule Generation with VAE" by Victor Basu (Apache License 2.0),
  which combines R. Gómez-Bombarelli et al., *ACS Central Science* 4, 268 (2018) and N. De Cao and T. Kipf, MolGAN
  (arXiv:1805.11973). Structure matching and oxidation states use pymatgen: S. P. Ong et al., *Computational Materials
  Science* 68, 314 (2013).